In [ ]:
!nvidia-smi
%pip install torchmetrics[image] -q

In [ ]:
from __future__ import annotations
from typing import Callable, Iterable, Optional

import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from torch.optim import Optimizer

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance

In [ ]:
SEED = 42 # For robust reporting, run multiple seeds: 42, 6, 11, 96, 22
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class HNAdam(Optimizer):
    def __init__(
        self,
        params: Iterable[Tensor],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.99),
        eps: float = 1e-8,
        lambda_t0: Optional[float] = None,
    ) -> None:
        if params is None:
            raise ValueError("params cannot be None.")
        if lr <= 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if len(betas) != 2:
            raise ValueError("betas must be a tuple of two floats")

        beta1, beta2 = betas
        if not 0.0 <= beta1 < 1.0:
            raise ValueError(f"Invalid beta1 value: {beta1}")
        if not 0.0 <= beta2 < 1.0:
            raise ValueError(f"Invalid beta2 value: {beta2}")

        if lambda_t0 is None:
            lambda_t0 = random.uniform(2.0, 4.0)
        if not 2.0 <= lambda_t0 <= 4.0:
            raise ValueError(f"lambda_t0 must be in [2, 4], got {lambda_t0}")

        defaults = {
            "lr": lr,
            "betas": (beta1, beta2),
            "eps": eps,
            "lambda_t0": lambda_t0,
            "amsgrad": False,
        }
        super().__init__(params, defaults)

        if len(self.param_groups) == 0:
            raise ValueError("optimizer got an empty parameter list")

    @torch.no_grad()
    def step(self, closure: Optional[Callable[[], Tensor]] = None) -> Optional[Tensor]:
        loss: Optional[Tensor] = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr: float = group["lr"]
            beta1, beta2 = group["betas"]
            eps: float = group["eps"]
            lambda_t0: float = group["lambda_t0"]

            for param in group["params"]:
                if param.grad is None:
                    continue

                grad = param.grad
                if grad.is_sparse:
                    raise RuntimeError("HNAdam does not support sparse gradients")

                state = self.state[param]

                if len(state) == 0:
                    state["m"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["v"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["vhat"] = torch.zeros_like(param, memory_format=torch.preserve_format)

                m_prev: Tensor = state["m"]
                v_prev: Tensor = state["v"]
                vhat_prev: Tensor = state["vhat"]

                g_t = grad

                m_t = beta1 * m_prev + (1.0 - beta1) * g_t

                g_abs = g_t.abs()
                m_prev_norm = torch.linalg.vector_norm(m_prev)
                g_abs_norm = torch.linalg.vector_norm(g_abs)
                m_max = torch.maximum(m_prev_norm, g_abs_norm)

                zero = torch.zeros((), dtype=param.dtype, device=param.device)
                ratio = torch.where(m_max > 0.0, m_prev_norm / m_max, zero)
                lambda_t = torch.as_tensor(lambda_t0, dtype=param.dtype, device=param.device) - ratio

                v_t = beta2 * v_prev + (1.0 - beta2) * g_abs.pow(lambda_t)

                if bool((lambda_t < 2.0).item()):
                    group["amsgrad"] = True

                    vhat_t = torch.maximum(vhat_prev, v_t.abs())
                    state["vhat"] = vhat_t

                    denom = vhat_t.pow(1.0 / lambda_t) + eps
                else:
                    group["amsgrad"] = False

                    denom = v_t.pow(1.0 / lambda_t) + eps

                param.addcdiv_(m_t, denom, value=-lr)

                state["m"] = m_t
                state["v"] = v_t

        return loss

In [ ]:
# Hyperparameters
input_dim = 784  # 28 x 28 images flattened
hidden_dim = 500
latent_dim = 20
condition_dim = 10  # FashionMNIST labels 0-9 for class conditioning
BATCH_SIZE = 128
EPOCHS = 50
lr = 1e-3
betas = (0.9, 0.99)
eps = 1e-8

# Data loader
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')

In [ ]:
# Define the Encoder, Decoder, and Conditional VAE
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, condition_dim):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim + condition_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)
        h = torch.tanh(self.fc1(xy))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, condition_dim):
        super(Decoder, self).__init__()
        self.fc1 = nn.Linear(latent_dim + condition_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z, y):
        zy = torch.cat([z, y], dim=1)
        h = torch.tanh(self.fc1(zy))
        x_hat = torch.sigmoid(self.fc2(h))
        return x_hat


class CVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, condition_dim):
        super(CVAE, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim, condition_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim, condition_dim)

    def reparameterize(self, mu, logvar):
        # Reparameterization trick: z = mu + epsilon * std
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mu + epsilon * std
        return z

    def forward(self, x, y):
        mu, logvar = self.encoder(x, y)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z, y)
        return x_hat, mu, logvar


def build_model():
    return CVAE(input_dim, hidden_dim, latent_dim, condition_dim).to(device)


build_model()

In [ ]:
# Define the loss function
def vae_loss(x, x_hat, mu, logvar):
    # Reconstruction term (Bernoulli likelihood)
    bce = F.binary_cross_entropy(x_hat, x, reduction='sum')

    # KL divergence between q(z|x) and p(z)=N(0,I)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return bce + kld


optimizer_builders = {
    'HNAdam': lambda params: HNAdam(params, lr=lr, betas=betas, eps=eps),
    'Adam': lambda params: torch.optim.Adam(params, lr=lr, betas=betas, eps=eps, amsgrad=False),
    'AMSGrad': lambda params: torch.optim.Adam(params, lr=lr, betas=betas, eps=eps, amsgrad=True),
    'SGD': lambda params: torch.optim.SGD(params, lr=lr),
}

print('Comparing optimizers:', ', '.join(optimizer_builders.keys()))

In [ ]:
histories = {}
models = {}
training_times = {}


def train_model(model, optimizer, train_loader, epochs):
    history = []
    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        running_total = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            y_onehot = F.one_hot(y, num_classes=condition_dim).float()
            x_flat = x.view(x.size(0), -1)

            optimizer.zero_grad()
            x_hat, mu, logvar = model(x_flat, y_onehot)
            total_loss = vae_loss(x_flat, x_hat, mu, logvar)
            if isinstance(optimizer, torch.optim.SGD):  # Only apply gradient clipping for SGD to prevent divergence
                (total_loss / x.size(0)).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            else:
                total_loss.backward()
            optimizer.step()

            running_total += total_loss.item()

        avg_total = running_total / len(train_loader.dataset)
        history.append(avg_total)
        print(f'Epoch {epoch:02d}/{epochs} | total={avg_total:.4f}')

    training_time_sec = time.perf_counter() - start_time
    return history, training_time_sec


for optimizer_name, optimizer_builder in optimizer_builders.items():
    print(f'\n===== Training with {optimizer_name} =====')
    model = build_model()
    optimizer = optimizer_builder(model.parameters())
    history, training_time_sec = train_model(model, optimizer, train_loader, EPOCHS)

    models[optimizer_name] = model
    histories[optimizer_name] = history
    training_times[optimizer_name] = training_time_sec

    print(f'{optimizer_name} completed in {training_time_sec:.2f} seconds')

In [ ]:
# Figure 1: loss minimization curves
epochs = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(10, 6))
for optimizer_name, loss_history in histories.items():
    plt.plot(epochs, loss_history, label=optimizer_name, linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Loss (average per sample)')
plt.title('Figure 1. VAE Loss Function Minimization Curves by Optimizer')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
@torch.no_grad()
def show_reconstructions(model, data_loader, title, n_images=10):
    model.eval()
    x, y = next(iter(data_loader))
    x = x.to(device)
    y = y.to(device)
    y_onehot = F.one_hot(y, num_classes=condition_dim).float()
    x_flat = x.view(x.size(0), -1)

    x_prob, _, _ = model(x_flat, y_onehot)
    x_recon = x_prob.view(-1, 1, 28, 28)

    n = min(n_images, x.size(0))
    _, axes = plt.subplots(2, n, figsize=(1.7 * n, 3.4))
    if n == 1:
        axes = np.array(axes).reshape(2, 1)

    for i in range(n):
        label = int(y[i].item())
        axes[0, i].imshow(x[i, 0].cpu(), cmap='gray')
        axes[0, i].set_title(f'y={label}', fontsize=9)
        axes[0, i].axis('off')
        axes[1, i].imshow(x_recon[i, 0].cpu(), cmap='gray')
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=10)
    axes[1, 0].set_ylabel('Reconstructed', fontsize=10)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


for optimizer_name, model in models.items():
    show_reconstructions(
        model,
        test_loader,
        title=f'{optimizer_name}: Original vs Reconstructed FashionMNIST images (conditioned on true labels)',
        n_images=10,
    )

In [ ]:
@torch.no_grad()
def show_per_class_generation_panel(
    model,
    latent_dim,
    condition_dim=10,
    samples_per_class=8,
    title='Per-class generated FashionMNIST samples (rows=labels 0-9)',
):
    model.eval()

    fig, axes = plt.subplots(
        condition_dim,
        samples_per_class,
        figsize=(samples_per_class * 1.15, condition_dim * 1.15),
    )

    if samples_per_class == 1:
        axes = np.expand_dims(axes, axis=1)

    for label in range(condition_dim):
        y = torch.full((samples_per_class,), label, device=device, dtype=torch.long)
        y_onehot = F.one_hot(y, num_classes=condition_dim).float()
        z = torch.randn(samples_per_class, latent_dim, device=device)

        probs = model.decoder(z, y_onehot).view(-1, 1, 28, 28)
        samples = torch.bernoulli(probs)

        for col in range(samples_per_class):
            axes[label, col].imshow(samples[col, 0].cpu(), cmap='gray')
            axes[label, col].axis('off')
            if col == 0:
                axes[label, col].set_ylabel(f'{label}', rotation=0, labelpad=12, va='center', fontsize=9)

    for col in range(samples_per_class):
        axes[0, col].set_title(f'S{col + 1}', fontsize=8)

    fig.text(0.01, 0.5, 'Label', va='center', rotation=90, fontsize=10)
    plt.suptitle(title, y=0.995)
    plt.tight_layout()
    plt.show()


for optimizer_name, model in models.items():
    show_per_class_generation_panel(
        model,
        latent_dim=latent_dim,
        condition_dim=condition_dim,
        samples_per_class=8,
        title=f'{optimizer_name}: Per-class generated FashionMNIST samples (rows=labels 0-9)',
    )

In [ ]:
@torch.no_grad()
def preprocess_for_fid(images):
    # FID expects 3-channel images, typically uint8 in [0, 255]
    images = images.repeat(1, 3, 1, 1)
    images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
    images = (images.clamp(0, 1) * 255).to(torch.uint8)
    return images


@torch.no_grad()
def evaluate_ssim_and_fid(model, test_loader, latent_dim=20, condition_dim=10, num_fid_samples=None):
    model.eval()

    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)

    total_test = len(test_loader.dataset)
    if num_fid_samples is None:
        num_fid_samples = total_test

    # Real and reconstructed images for SSIM
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)
        y_onehot = F.one_hot(y, num_classes=condition_dim).float()
        x_flat = x.view(x.size(0), -1)
        x_prob, _, _ = model(x_flat, y_onehot)
        x_recon = x_prob.view(-1, 1, 28, 28)

        ssim_metric.update(x_recon, x)

    ssim_value = float(ssim_metric.compute().item())

    # FID: real test images vs generated images
    real_seen = 0
    for x, _ in test_loader:
        if real_seen >= num_fid_samples:
            break

        x = x.to(device)
        needed = min(x.size(0), num_fid_samples - real_seen)
        real_batch = x[:needed]
        fid_metric.update(preprocess_for_fid(real_batch), real=True)
        real_seen += needed

    fake_seen = 0
    while fake_seen < num_fid_samples:
        current_bs = min(BATCH_SIZE, num_fid_samples - fake_seen)
        z = torch.randn(current_bs, latent_dim, device=device)
        y = torch.randint(0, condition_dim, (current_bs,), device=device)
        y_onehot = F.one_hot(y, num_classes=condition_dim).float()
        x_gen_prob = model.decoder(z, y_onehot).view(-1, 1, 28, 28)
        x_gen = torch.bernoulli(x_gen_prob)
        fid_metric.update(preprocess_for_fid(x_gen), real=False)
        fake_seen += current_bs

    fid_value = float(fid_metric.compute().item())
    return ssim_value, fid_value


comparison_rows = []
num_fid_samples = 10000

for optimizer_name, model in models.items():
    ssim_value, fid_value = evaluate_ssim_and_fid(
        model,
        test_loader,
        latent_dim=latent_dim,
        condition_dim=condition_dim,
        num_fid_samples=num_fid_samples,
    )

    comparison_rows.append({
        'Model': f'Conditional VAE x FashionMNIST x {optimizer_name}',
        'Optimizer': optimizer_name,
        'Min Training Loss': float(np.min(histories[optimizer_name])),
        'Training Time (s)': float(training_times[optimizer_name]),
        'SSIM': ssim_value,
        'FID': fid_value,
    })

    print(f'{optimizer_name} -> SSIM: {ssim_value:.4f} | FID: {fid_value:.4f}')

comparison_df = pd.DataFrame(comparison_rows)

In [ ]:
# Table 1: summary statistics across optimizers
table_1 = comparison_df.copy()

table_1 = table_1.round({
    'Min Training Loss': 4,
    'Training Time (s)': 2,
    'SSIM': 4,
    'FID': 4
})

optimizer_order = ['HNAdam', 'Adam', 'AMSGrad', 'SGD']
table_1['Optimizer'] = pd.Categorical(table_1['Optimizer'], categories=optimizer_order, ordered=True)
table_1 = table_1.sort_values(by='Optimizer').reset_index(drop=True)

print('Table 1. Training and Evaluation Summary by Optimizer')
display(table_1)